In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
"""
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
"""

In [ ]:
# =============================================================================
# cell 1 setup
# traditional_descriptors_kaggle.ipynb
# Traditional feature extraction — SIFT, ORB, Color Histograms
# Assigned to: Mahnoor
#
# HOW TO RUN:
# 1. Attach dataset: kaggle.com/datasets/sohangundoju/mirflickr-1m
# 2. Settings → Accelerator → None  (CPU task, no GPU needed)
# 3. Only change PACK_NUMBER each session. Nothing else.
# =============================================================================

PACK_NUMBER     = 1       # ← Change this to 1, 2, 3 ... 10 each session
IMAGES_PER_PACK = 100_000  # do not change
KAGGLE_OUTPUT   = '/kaggle/working'

# MUST match Piranchal's and Aliza's path pattern exactly
KAGGLE_INPUT = f'/kaggle/input/datasets/sohangundoju/mirflickr-1m/images{PACK_NUMBER - 1}/images'

print(f"Pack {PACK_NUMBER} config loaded.")
print(f"  Input  : {KAGGLE_INPUT}")
print(f"  Output : {KAGGLE_OUTPUT}")

In [ ]:
# cell 2 dependencies
!pip install opencv-contrib-python-headless tqdm --quiet

In [ ]:
# cell 3 imports
import os
import glob
import pickle
import multiprocessing

import cv2
import numpy as np
from tqdm import tqdm

print("✓ All imports successful")
print(f"  OpenCV : {cv2.__version__}")
print(f"  NumPy  : {np.__version__}")
print(f"  CPUs   : {os.cpu_count()}")

In [ ]:
# CELL 4 — SET UP OUTPUT PATHS

TRAD_OUT_DIR = os.path.join(KAGGLE_OUTPUT, 'features', 'traditional')
SIFT_OUT_DIR = os.path.join(TRAD_OUT_DIR, 'sift')
ORB_OUT_DIR  = os.path.join(TRAD_OUT_DIR, 'orb')
HIST_OUT_DIR = os.path.join(TRAD_OUT_DIR, 'hist')

for d in [SIFT_OUT_DIR, ORB_OUT_DIR, HIST_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

sift_pkl = os.path.join(SIFT_OUT_DIR, f'pack_{PACK_NUMBER}_sift.pkl')
orb_pkl  = os.path.join(ORB_OUT_DIR,  f'pack_{PACK_NUMBER}_orb.pkl')
hist_pkl = os.path.join(HIST_OUT_DIR, f'pack_{PACK_NUMBER}_hist.pkl')

print(f"✓ Output paths set.")
print(f"  SIFT : {sift_pkl}")
print(f"  ORB  : {orb_pkl}")
print(f"  HIST : {hist_pkl}")

In [ ]:
# CELL 5 — CHECK IF THIS PACK IS ALREADY DONE

if os.path.exists(sift_pkl) and os.path.exists(orb_pkl) and os.path.exists(hist_pkl):
    print(f"✓ Pack {PACK_NUMBER} already extracted:")
    print(f"  {sift_pkl}")
    print(f"  {orb_pkl}")
    print(f"  {hist_pkl}")
    print("  Change PACK_NUMBER in Cell 1 to process next pack.")
    raise SystemExit("Already processed.")

print(f"Pack {PACK_NUMBER} not yet processed. Proceeding.")

In [ ]:
# CELL 6 — FIND IMAGES FOR THIS PACK
# index 0 here == index 0 in CNN embeddings and hashes.

print(f"Scanning Pack {PACK_NUMBER} at {KAGGLE_INPUT} ...")

all_patterns    = ['**/*.jpg', '**/*.jpeg', '**/*.png']
all_image_paths = []
for pattern in all_patterns:
    all_image_paths.extend(
        glob.glob(os.path.join(KAGGLE_INPUT, pattern), recursive=True)
    )

# CRITICAL: identical dedup + sort to Piranchal and Aliza
PACK_PATHS = sorted(list(set([os.path.abspath(p) for p in all_image_paths])))
pack_start = (PACK_NUMBER - 1) * IMAGES_PER_PACK
pack_end   = pack_start + len(PACK_PATHS)

print(f"✓ Pack {PACK_NUMBER}:")
print(f"  Images  : {len(PACK_PATHS)}")
print(f"  ID range: {pack_start} → {pack_end - 1}")
print(f"  First   : {os.path.basename(PACK_PATHS[0])}")
print(f"  Last    : {os.path.basename(PACK_PATHS[-1])}")

if len(PACK_PATHS) == 0:
    raise ValueError(
        f"No images found at {KAGGLE_INPUT}. "
        "Check dataset is attached and PACK_NUMBER is correct."
    )

if len(PACK_PATHS) < 90_000:
    print(f"  WARNING: Only {len(PACK_PATHS)} images found — expected ~100,000.")

In [ ]:
# CELL 7 — SYNC CHECK
# Share this output in the group chat.
# Piranchal and Aliza run the same cell — if filenames differ, stop and debug.

print("=" * 60)
print(f"  SYNC CHECK — Pack {PACK_NUMBER} — share this in group chat")
print("=" * 60)
print(f"  Pack images : {len(PACK_PATHS)}")
print(f"  ID range    : {pack_start} → {pack_end - 1}")
print(f"  First file  : {os.path.basename(PACK_PATHS[0])}")
print(f"  Last  file  : {os.path.basename(PACK_PATHS[-1])}")
if len(PACK_PATHS) > 500:
    print(f"  File #500   : {os.path.basename(PACK_PATHS[500])}")
if len(PACK_PATHS) > 50_000:
    print(f"  File #50000 : {os.path.basename(PACK_PATHS[50_000])}")
print("=" * 60)
print("  If any filename differs from Piranchal/Aliza — stop and debug.")
print("=" * 60)

In [ ]:
# CELL 8 — FEATURE EXTRACTION FUNCTIONS
#
# SIFT : 128-dim float32  (mean-pooled across all keypoints)
# ORB  : 32-dim  float32  (mean-pooled; Hamming-space — note for fusion step)
# HIST : 96-dim  float32  (2×2 grid, 3 RGB channels, 8 bins each, L1-normalized)
#
# Zero vectors returned for blank/unreadable images — never crashes.

SIFT_DIM      = 128
ORB_DIM       = 32
HIST_DIM      = 96      # 4 regions × 3 channels × 8 bins
MAX_KEYPOINTS = 500     # cap for ORB (per task spec)
N_WORKERS     = os.cpu_count()


def extract_all_features(args):
    """
    Worker function — runs in a subprocess (multiprocessing, not threading).
    CPU-bound work: multiprocessing bypasses the GIL properly.

    args : (image_id, img_path_str)
    returns : (image_id, sift_vec, orb_vec, hist_vec)  — all float32 numpy arrays
    """
    image_id, img_path = args

    # ── Read ──────────────────────────────────────────────────────────────────
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return (image_id,
                np.zeros(SIFT_DIM, dtype=np.float32),
                np.zeros(ORB_DIM,  dtype=np.float32),
                np.zeros(HIST_DIM, dtype=np.float32))

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # ── SIFT ──────────────────────────────────────────────────────────────────
    try:
        sift = cv2.SIFT_create()
        _, sift_descs = sift.detectAndCompute(gray, None)
        if sift_descs is not None and len(sift_descs) > 0:
            sift_vec = sift_descs.mean(axis=0).astype(np.float32)
        else:
            sift_vec = np.zeros(SIFT_DIM, dtype=np.float32)
    except Exception:
        sift_vec = np.zeros(SIFT_DIM, dtype=np.float32)

    # ── ORB ───────────────────────────────────────────────────────────────────
    try:
        orb = cv2.ORB_create(nfeatures=MAX_KEYPOINTS)
        _, orb_descs = orb.detectAndCompute(gray, None)
        if orb_descs is not None and len(orb_descs) > 0:
            orb_vec = orb_descs.astype(np.float32).mean(axis=0)
        else:
            orb_vec = np.zeros(ORB_DIM, dtype=np.float32)
    except Exception:
        orb_vec = np.zeros(ORB_DIM, dtype=np.float32)

    # ── Color Histogram — 2×2 grid, RGB, 8 bins/channel ──────────────────────
    try:
        h, w   = img_bgr.shape[:2]
        h2, w2 = h // 2, w // 2
        regions = [
            img_bgr[:h2, :w2],   # top-left
            img_bgr[:h2, w2:],   # top-right
            img_bgr[h2:, :w2],   # bottom-left
            img_bgr[h2:, w2:],   # bottom-right
        ]
        hist_parts = []
        for region in regions:
            if region.size == 0:
                hist_parts.append(np.zeros(24, dtype=np.float32))
                continue
            region_hist = []
            for ch in range(3):                          # B, G, R
                ch_hist = cv2.calcHist(
                    [region], [ch], None, [8], [0, 256]
                ).flatten().astype(np.float32)
                total = ch_hist.sum()
                if total > 0:
                    ch_hist /= total                     # L1 normalize per channel
                region_hist.append(ch_hist)
            hist_parts.append(np.concatenate(region_hist))  # 24-dim per region
        hist_vec = np.concatenate(hist_parts)                # 96-dim total
    except Exception:
        hist_vec = np.zeros(HIST_DIM, dtype=np.float32)

    return (image_id, sift_vec, orb_vec, hist_vec)


print(f"✓ Extraction functions defined.")
print(f"  SIFT dim : {SIFT_DIM}   (mean-pooled float32)")
print(f"  ORB  dim : {ORB_DIM}    (mean-pooled float32, Hamming-space)")
print(f"  HIST dim : {HIST_DIM}   (2×2 grid, RGB, 8 bins, L1-norm)")
print(f"  Workers  : {N_WORKERS}")

In [ ]:
# CELL 9 — BUILD ARGS LIST

args_list = [
    (pack_start + i, path)
    for i, path in enumerate(PACK_PATHS)
]

print(f"✓ Args list ready — {len(args_list):,} images")
print(f"  First ID : {args_list[0][0]}  →  {os.path.basename(args_list[0][1])}")
print(f"  Last  ID : {args_list[-1][0]} →  {os.path.basename(args_list[-1][1])}")

In [ ]:
# CELL 10 — RUN PARALLEL EXTRACTION
# multiprocessing.Pool — CPU-bound work, bypasses the GIL.
# We use ProcessPool because SIFT/ORB are pure CPU computation.

import time

sift_features = {}
orb_features  = {}
hist_features = {}
zero_count    = 0
t0            = time.time()

print(f"Starting extraction on {len(args_list):,} images with {N_WORKERS} workers ...")
print("(SIFT + ORB + Histogram computed per image in a single parallel pass)")

with multiprocessing.Pool(processes=N_WORKERS) as pool:
    for image_id, sift_vec, orb_vec, hist_vec in tqdm(
        pool.imap(extract_all_features, args_list, chunksize=64),
        total=len(args_list),
        desc=f"Pack {PACK_NUMBER}",
        unit="img"
    ):
        sift_features[image_id] = sift_vec
        orb_features[image_id]  = orb_vec
        hist_features[image_id] = hist_vec
        if np.all(sift_vec == 0) and np.all(orb_vec == 0):
            zero_count += 1

elapsed = time.time() - t0
print(f"\n✓ Extraction complete in {elapsed/60:.1f} min")
print(f"  Images processed : {len(sift_features):,}")
print(f"  Failed images    : {zero_count}  (blank/unreadable — expected ~0)")

In [ ]:
# CELL 11 — SAVE OUTPUT

with open(sift_pkl, 'wb') as f: pickle.dump(sift_features, f)
with open(orb_pkl,  'wb') as f: pickle.dump(orb_features,  f)
with open(hist_pkl, 'wb') as f: pickle.dump(hist_features, f)

print(f"✓ Saved:")
print(f"  {sift_pkl}  ({os.path.getsize(sift_pkl)/1e6:.1f} MB)")
print(f"  {orb_pkl}   ({os.path.getsize(orb_pkl)/1e6:.1f} MB)")
print(f"  {hist_pkl}  ({os.path.getsize(hist_pkl)/1e6:.1f} MB)")

In [ ]:
# CELL 12 — VERIFY OUTPUT

print("Running verification checks ...\n")

assert len(sift_features) == len(PACK_PATHS), \
    f"SIFT count mismatch: {len(sift_features)} vs {len(PACK_PATHS)}"
assert len(orb_features)  == len(PACK_PATHS), \
    f"ORB  count mismatch: {len(orb_features)} vs {len(PACK_PATHS)}"
assert len(hist_features) == len(PACK_PATHS), \
    f"HIST count mismatch: {len(hist_features)} vs {len(PACK_PATHS)}"
print("✓ Count check passed")

sample_id = pack_start
assert sift_features[sample_id].shape == (SIFT_DIM,), "SIFT shape wrong"
assert orb_features[sample_id].shape  == (ORB_DIM,),  "ORB  shape wrong"
assert hist_features[sample_id].shape == (HIST_DIM,), "HIST shape wrong"
print(f"✓ Shape check passed — SIFT({SIFT_DIM},) ORB({ORB_DIM},) HIST({HIST_DIM},)")

your_ids = sorted(sift_features.keys())
assert your_ids[0]  == pack_start,    f"First ID wrong: {your_ids[0]}"
assert your_ids[-1] == pack_end - 1,  f"Last  ID wrong: {your_ids[-1]}"
print(f"✓ ID alignment passed — {your_ids[0]} → {your_ids[-1]}")

assert sift_features[sample_id].dtype == np.float32, "SIFT not float32"
assert orb_features[sample_id].dtype  == np.float32, "ORB  not float32"
assert hist_features[sample_id].dtype == np.float32, "HIST not float32"
print("✓ dtype check passed — all float32")

zero_sift = sum(1 for v in sift_features.values() if np.all(v == 0))
zero_pct  = zero_sift / len(sift_features) * 100
print(f"✓ Zero-vector rate: {zero_sift:,} ({zero_pct:.2f}%)")
if zero_pct > 5:
    print("  WARNING: >5% zero vectors — check if images are readable")

print()
print("=" * 60)
print(f"  Pack {PACK_NUMBER} COMPLETE")
print(f"  Files saved to /kaggle/working/features/traditional/")
print()
print(f"  ► Download from Output panel (right sidebar):")
print(f"    sift/pack_{PACK_NUMBER}_sift.pkl")
print(f"    orb/pack_{PACK_NUMBER}_orb.pkl")
print(f"    hist/pack_{PACK_NUMBER}_hist.pkl")
print()
print(f"  ► Change PACK_NUMBER to {PACK_NUMBER + 1} for next session")
print("=" * 60)

In [ ]:
import os

for name in os.listdir('/kaggle/input'):
    path = f'/kaggle/input/{name}'
    print(f"\n{path}")
    try:
        files = os.listdir(path)
        for f in sorted(files):
            print(f"  {f}")
    except:
        print("  (unreadable)")

In [ ]:
# CELL 13 — MERGE (3 SEPARATE DATASETS)

import os, pickle

SIFT_DIR = '/kaggle/input/trad-siftt'
ORB_DIR  = '/kaggle/input/trad-orbb'
HIST_DIR = '/kaggle/input/trad-hist'

# Confirm all 3 datasets are visible
print("SIFT files:", sorted(os.listdir(SIFT_DIR)))
print("ORB  files:", sorted(os.listdir(ORB_DIR)))
print("HIST files:", sorted(os.listdir(HIST_DIR)))

all_sift = {}
all_orb  = {}
all_hist = {}

for n in range(1, 11):
    with open(os.path.join(SIFT_DIR, f'pack_{n}_sift.pkl'), 'rb') as f:
        all_sift.update(pickle.load(f))
    with open(os.path.join(ORB_DIR,  f'pack_{n}_orb.pkl'),  'rb') as f:
        all_orb.update(pickle.load(f))
    with open(os.path.join(HIST_DIR, f'pack_{n}_hist.pkl'), 'rb') as f:
        all_hist.update(pickle.load(f))
    print(f"  Pack {n} loaded.")

print(f"\n✓ Merge complete.")
print(f"  SIFT : {len(all_sift):,}")
print(f"  ORB  : {len(all_orb):,}")
print(f"  HIST : {len(all_hist):,}")

with open('/kaggle/working/all_sift_features.pkl', 'wb') as f: pickle.dump(all_sift, f)
with open('/kaggle/working/all_orb_features.pkl',  'wb') as f: pickle.dump(all_orb,  f)
with open('/kaggle/working/all_hist_features.pkl', 'wb') as f: pickle.dump(all_hist, f)

print("\n✓ Saved to /kaggle/working/")
print("  Download: all_sift_features.pkl, all_orb_features.pkl, all_hist_features.pkl")
print("  Share all 3 with Piranchal for the fusion step.")